# ML Testing & MLOps: Catching Silent Model Failure Before It Ships

**Dataset:** [Wine Quality (UCI)](https://archive.ics.uci.edu/dataset/186/wine+quality) — two real
datasets of physicochemical lab measurements and expert sensory-panel quality scores for Portuguese
*vinho verde* wines: 1,599 **red** wine samples (loaded from a public CSV mirror of the UCI dataset)
and 4,898 **white** wine samples (loaded from a second public CSV mirror). Same 11 real-valued
chemistry features (acidity, sugar, sulphates, alcohol, etc.) and the same 0–10 quality score in both.

## Problem statement

You are the QA/MLOps engineer for a team that trained a binary classifier — "is this wine *good*?"
(quality score ≥ 7 vs. not) — on **red wine** data only. The model passed its offline evaluation and
is about to go live. Before it ships, your job is to build the automated test suite that a real MLOps
pipeline would run, because a model that looks fine on an aggregate metric can still fail badly in
production without anyone noticing until it's too late.

Concretely, after it ships, the model starts scoring a new incoming batch of wine samples — the
**white wine** dataset standing in for "the next batch of production traffic." You don't get to see
the white wine labels as a human in the loop; the test suite has to flag trouble on its own.

**What you should produce:**

1. `validate_batch(df)` — a data validation function that checks any incoming batch (red *or* white)
   against a fixed schema: required columns present, no unexpected columns, no nulls, no duplicate
   rows, and every feature within a physically plausible range. It should return a list of human-
   readable issue strings (empty list = batch passes).
2. A trained baseline classifier (`RandomForestClassifier`) evaluated on a held-out **red-wine** test
   split with precision/recall/F1 — plus a **slice-based evaluation** (`evaluate_slices`) that breaks
   that same test set into alcohol-content tertiles and flags any slice whose recall falls below a
   threshold. A single aggregate score can hide a slice where the model is far worse.
3. `detect_drift(reference_df, incoming_df, features)` — a drift detector that runs a two-sample
   Kolmogorov–Smirnov test per feature between the red-wine **training** distribution and the
   white-wine **incoming batch**, flagging any feature with p < 0.05.
4. A final comparison: run `detect_drift` on the white batch *before* touching its labels, then
   measure the model's *actual* precision/recall/F1 on the white batch using its real labels, and
   write one sentence on whether the drift signal would have been an early warning worth acting on.


---

## Starter cells

Fill in every `# TODO`. Don't open the solution until you've tried each step yourself.


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

RANDOM_STATE = 42

RED_URL = "https://raw.githubusercontent.com/plotly/datasets/master/winequality-red.csv"
WHITE_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/winequality-white.csv"

red = pd.read_csv(RED_URL, encoding="utf-8-sig")
white = pd.read_csv(WHITE_URL)
white.columns = white.columns.str.replace('"', "")

FEATURES = [c for c in red.columns if c != "quality"]

# Binary target: a "good" wine scores 7 or higher out of 10.
red["is_good"] = (red["quality"] >= 7).astype(int)
white["is_good"] = (white["quality"] >= 7).astype(int)

print(red.shape, white.shape)
red.head()


In [ ]:
# TODO: define SCHEMA, a dict mapping each column in FEATURES to a (low, high) tuple of
# physically plausible bounds (look at red.describe() / white.describe() to pick sane ranges —
# they must be wide enough to accept BOTH red and white wine without being so wide they catch
# nothing, e.g. pH roughly 2.5-4.5, alcohol roughly 7-15).

SCHEMA = {
    # "fixed acidity": (lo, hi),
    ...  # TODO: fill in all 11 features
}


In [ ]:
# TODO: implement validate_batch(df, schema=SCHEMA) -> list[str]
#
# It should check, and append a readable string to `issues` for each problem found:
#   - any schema column missing from df.columns                 -> "MISSING COLUMN: ..."
#   - any column in df that isn't in schema and isn't quality/is_good -> "UNEXPECTED COLUMNS: ..."
#   - any null values in the feature columns                      -> "NULLS: ..."
#   - any fully duplicated rows (df.duplicated())                  -> "DUPLICATES: ..."
#   - for each schema column, any rows outside [lo, hi]            -> "RANGE: <col> has N rows outside [lo, hi]"
# Return `issues` (empty list means the batch passed).

def validate_batch(df, schema=SCHEMA):
    issues = []
    # TODO
    return issues

print("red issues:", validate_batch(red))
print("white issues:", validate_batch(white))


In [ ]:
# TODO: split red into X_train, X_test, y_train, y_test using FEATURES / "is_good",
# test_size=0.25, stratify=y, random_state=RANDOM_STATE.
# Train a RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced")
# on the training split, predict on X_test, and print precision/recall/F1 on the test split.

X_train, X_test, y_train, y_test = ...  # TODO
clf = ...  # TODO
test_preds = ...  # TODO


In [ ]:
# TODO: implement evaluate_slices(y_true, y_pred, slice_series, min_recall=0.5) -> pd.DataFrame
#
# For each distinct value/category in slice_series:
#   - build a boolean mask for rows in that slice
#   - skip slices with 0 rows
#   - compute precision_score / recall_score on y_true[mask], y_pred[mask] (zero_division=0)
#   - a row dict: {"slice": label, "n": mask.sum(), "precision": p, "recall": r, "flag": r < min_recall}
# Return a DataFrame of all rows.
#
# Then: bin X_test["alcohol"] into 3 tertiles with pd.qcut(..., 3, labels=["low","mid","high"])
# and call evaluate_slices on the test predictions. Which slice gets flagged, and why might that
# matter for a wine retailer using this model?

def evaluate_slices(y_true, y_pred, slice_series, min_recall=0.5):
    rows = []
    # TODO
    return pd.DataFrame(rows)

alcohol_bins = ...  # TODO
slice_report = ...  # TODO
slice_report


In [ ]:
# TODO: implement detect_drift(reference_df, incoming_df, features, alpha=0.05) -> pd.DataFrame
#
# For each feature, run scipy.stats.ks_2samp(reference_df[feature], incoming_df[feature]) and
# build a row: {"feature": feature, "ks_stat": stat, "p_value": pvalue, "drift": pvalue < alpha}.
# Return a DataFrame sorted by ks_stat descending.
#
# Then call it with reference_df=X_train, incoming_df=white, features=FEATURES.
# How many of the 11 features show significant drift?

def detect_drift(reference_df, incoming_df, features, alpha=0.05):
    rows = []
    # TODO
    return pd.DataFrame(rows)

drift_report = ...  # TODO
drift_report


In [ ]:
# TODO: now measure what actually happens. Predict on white[FEATURES] with your trained `clf`
# and print precision/recall/F1 against white["is_good"] (the real labels, which a production
# system wouldn't have yet when the drift report ran).
#
# Compare this to the red-wine test performance from a few cells up, and to the drift report.
# In a markdown cell or comment, write one sentence: would the drift signal have been a useful
# early warning here, given you didn't need the labels to compute it?

white_preds = ...  # TODO


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

```python
SCHEMA = {
    "fixed acidity": (3.0, 16.0),
    "volatile acidity": (0.0, 2.0),
    "citric acid": (0.0, 2.0),
    "residual sugar": (0.0, 70.0),
    "chlorides": (0.0, 1.0),
    "free sulfur dioxide": (0.0, 300.0),
    "total sulfur dioxide": (0.0, 450.0),
    "density": (0.98, 1.04),
    "pH": (2.5, 4.5),
    "sulphates": (0.0, 2.5),
    "alcohol": (7.0, 15.0),
}

def validate_batch(df, schema=SCHEMA):
    issues = []
    for col in schema:
        if col not in df.columns:
            issues.append(f"MISSING COLUMN: {col}")
    extra = set(df.columns) - set(schema) - {"quality", "is_good"}
    if extra:
        issues.append(f"UNEXPECTED COLUMNS: {sorted(extra)}")
    n_nulls = df[list(schema)].isna().sum().sum()
    if n_nulls:
        issues.append(f"NULLS: {n_nulls} missing values across feature columns")
    n_dupes = df.duplicated().sum()
    if n_dupes:
        issues.append(f"DUPLICATES: {n_dupes} duplicate rows")
    for col, (lo, hi) in schema.items():
        if col not in df.columns:
            continue
        out = df[(df[col] < lo) | (df[col] > hi)]
        if len(out):
            issues.append(f"RANGE: {col} has {len(out)} rows outside [{lo}, {hi}]")
    return issues

print("red issues:", validate_batch(red))     # ['DUPLICATES: 240 duplicate rows']
print("white issues:", validate_batch(white)) # ['DUPLICATES: 937 duplicate rows']

X_train, X_test, y_train, y_test = train_test_split(
    red[FEATURES], red["is_good"], test_size=0.25, stratify=red["is_good"], random_state=RANDOM_STATE
)
clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced")
clf.fit(X_train, y_train)
test_preds = clf.predict(X_test)
print(f"red-test  precision={precision_score(y_test, test_preds):.3f} "
      f"recall={recall_score(y_test, test_preds):.3f} f1={f1_score(y_test, test_preds):.3f}")
# precision=0.741 recall=0.741 f1=0.741

def evaluate_slices(y_true, y_pred, slice_series, min_recall=0.5):
    rows = []
    categories = slice_series.cat.categories if hasattr(slice_series, "cat") else sorted(slice_series.unique())
    for label in categories:
        mask = (slice_series == label).values
        if mask.sum() == 0:
            continue
        p = precision_score(y_true[mask], y_pred[mask], zero_division=0)
        r = recall_score(y_true[mask], y_pred[mask], zero_division=0)
        rows.append({"slice": label, "n": int(mask.sum()), "precision": p, "recall": r, "flag": r < min_recall})
    return pd.DataFrame(rows)

alcohol_bins = pd.qcut(X_test["alcohol"], 3, labels=["low", "mid", "high"])
slice_report = evaluate_slices(y_test.values, test_preds, alcohol_bins, min_recall=0.5)
print(slice_report)
#  slice    n  precision    recall   flag
#    low  135   0.000000  0.000000   True
#    mid  134   1.000000  0.250000   True
#   high  131   0.725490  0.948718  False

def detect_drift(reference_df, incoming_df, features, alpha=0.05):
    rows = []
    for col in features:
        stat, pvalue = stats.ks_2samp(reference_df[col], incoming_df[col])
        rows.append({"feature": col, "ks_stat": stat, "p_value": pvalue, "drift": pvalue < alpha})
    return pd.DataFrame(rows).sort_values("ks_stat", ascending=False)

drift_report = detect_drift(X_train, white, FEATURES)
print(drift_report)
print(f"{drift_report['drift'].sum()} / {len(drift_report)} features drift")  # 11 / 11

white_preds = clf.predict(white[FEATURES])
print(f"white precision={precision_score(white['is_good'], white_preds):.3f} "
      f"recall={recall_score(white['is_good'], white_preds):.3f} "
      f"f1={f1_score(white['is_good'], white_preds):.3f}")
# precision=0.514 recall=0.160 f1=0.244
```

**Why this matters for ML quality engineering:** `validate_batch` is the ML equivalent of a schema/
contract test on an API response — it catches structurally broken input before it ever reaches the
model. `evaluate_slices` shows why "74% F1 overall" is not the whole story: the low-alcohol slice has
*zero* recall, meaning the model never once catches a good low-alcohol wine in that slice — a failure
an aggregate metric completely hides. `detect_drift` is a monitoring check that needs **no labels at
all**, which is exactly why it matters: in production you often don't get ground truth for days or
weeks, but you get new feature data immediately. Here all 11 features drift significantly between red
and white wine, and sure enough, real performance on the white batch collapses from an F1 of 0.741 to
0.244 — the drift alarm would have fired *before* that collapse was ever confirmed by labels, which is
the whole point of monitoring feature distributions instead of waiting on a lagging performance metric.

</details>
